In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [4]:
import numpy as np
import pandas as pd
import os
import warnings
import joblib
import gc

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

import xgboost as xgb
import mlflow
import dagshub

warnings.filterwarnings("ignore")
print("All imports OK ✓")

All imports OK ✓


**load and merge data. clean, reduce memory**

In [5]:
path = "/kaggle/input/competitions/ieee-fraud-detection/"

print("Loading data...")
train_tr = pd.read_csv(path + "train_transaction.csv")
train_id = pd.read_csv(path + "train_identity.csv")
test_tr  = pd.read_csv(path + "test_transaction.csv")
test_id  = pd.read_csv(path + "test_identity.csv")
print(f"  train_transaction: {train_tr.shape}")
print(f"  train_identity:    {train_id.shape}")
print(f"  test_transaction:  {test_tr.shape}")
print(f"  test_identity:     {test_id.shape}")

print("\nFixing column names (hyphen → underscore)...")
train_id.columns = [c.replace("-", "_") for c in train_id.columns]
test_id.columns  = [c.replace("-", "_") for c in test_id.columns]

print("\nMerging...")
train = train_tr.merge(train_id, on="TransactionID", how="left")
test  = test_tr.merge(test_id,   on="TransactionID", how="left")
print(f"  train merged: {train.shape}")
print(f"  test merged:  {test.shape}")

X = train.drop(columns=["isFraud", "TransactionID"])
y = train["isFraud"]
test = test.drop(columns=["TransactionID"])

print("\nAligning columns...")
only_in_train = set(X.columns) - set(test.columns)
only_in_test  = set(test.columns) - set(X.columns)
print(f"  Cols only in train: {len(only_in_train)}")
print(f"  Cols only in test:  {len(only_in_test)}")
common_cols = [c for c in X.columns if c in test.columns]
X    = X[common_cols]
test = test[common_cols]
assert list(X.columns) == list(test.columns)
print(f"  Aligned: X={X.shape}, test={test.shape}")
print(f"  Fraud rate: {y.mean():.4f} ({y.sum()} fraud out of {len(y)} total)")

Loading data...
  train_transaction: (590540, 394)
  train_identity:    (144233, 41)
  test_transaction:  (506691, 393)
  test_identity:     (141907, 41)

Fixing column names (hyphen → underscore)...

Merging...
  train merged: (590540, 434)
  test merged:  (506691, 433)

Aligning columns...
  Cols only in train: 0
  Cols only in test:  0
  Aligned: X=(590540, 432), test=(506691, 432)
  Fraud rate: 0.0350 (20663 fraud out of 590540 total)


In [6]:
print("=== FEATURE ENGINEERING ===")
print(f"Starting shape: X={X.shape}, test={test.shape}")

def engineer_features(df):
    # in-place, no copy

    # ── 1. TransactionAmt features ──
    df["TransactionAmt_log"]      = np.log1p(df["TransactionAmt"]).astype(np.float32)
    df["TransactionAmt_is_round"] = (df["TransactionAmt"] % 1 == 0).astype(np.int8)
    df["TransactionAmt_cents"]    = (df["TransactionAmt"] % 1).astype(np.float32)

    # ── 2. time features ──
    df["hour"] = ((df["TransactionDT"] / 3600) % 24).astype(np.float32)
    df["day"]  = ((df["TransactionDT"] / 86400) % 7).astype(np.float32)

    # ── 3. email provider ──
    if "P_emaildomain" in df.columns:
        df["P_email_provider"] = df["P_emaildomain"].str.split(".").str[0].fillna("unknown")
    if "R_emaildomain" in df.columns:
        df["R_email_provider"] = df["R_emaildomain"].str.split(".").str[0].fillna("unknown")

    # ── 4. card identity ──
    df["card_identity"] = (
        df["card1"].astype(str) + "_" +
        df["card2"].astype(str) + "_" +
        df["card4"].astype(str)
    )

    # ── 5. C columns aggregates ──
    c_cols = [c for c in df.columns if c.startswith("C") and c[1:].isdigit()]
    if len(c_cols) > 0:
        print(f"  C columns found: {c_cols}")
        df["C_mean"] = df[c_cols].mean(axis=1).astype(np.float32)
        df["C_std"]  = df[c_cols].std(axis=1).astype(np.float32)
        df["C_max"]  = df[c_cols].max(axis=1).astype(np.float32)
        df["C_min"]  = df[c_cols].min(axis=1).astype(np.float32)

# ── compute card aggregates from train only ──
print("Computing card1 aggregates from train...")
card1_count    = X["card1"].value_counts()
card1_amt_mean = X.groupby("card1")["TransactionAmt"].mean()
card1_amt_std  = X.groupby("card1")["TransactionAmt"].std()
print(f"  Unique card1 values: {len(card1_count)}")

def add_card_aggregates(df):
    df["card1_count"]    = df["card1"].map(card1_count).astype(np.float32)
    df["card1_amt_mean"] = df["card1"].map(card1_amt_mean).astype(np.float32)
    df["card1_amt_std"]  = df["card1"].map(card1_amt_std).astype(np.float32)

print("Engineering X features...")
engineer_features(X)
add_card_aggregates(X)
gc.collect()
print(f"  X shape: {X.shape}")
print(f"  X memory: {X.memory_usage(deep=True).sum()/1024**2:.1f} MB")

print("Engineering test features...")
engineer_features(test)
add_card_aggregates(test)
gc.collect()
print(f"  test shape: {test.shape}")
print(f"  test memory: {test.memory_usage(deep=True).sum()/1024**2:.1f} MB")

# realign
print("\nRealigning columns...")
common_cols = [c for c in X.columns if c in test.columns]
X    = X[common_cols]
test = test[common_cols]
assert list(X.columns) == list(test.columns)
print(f"Final: X={X.shape}, test={test.shape}")
print("Feature engineering done ✓")

=== FEATURE ENGINEERING ===
Starting shape: X=(590540, 432), test=(506691, 432)
Computing card1 aggregates from train...
  Unique card1 values: 13553
Engineering X features...
  C columns found: ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14']
  X shape: (590540, 447)
  X memory: 2630.0 MB
Engineering test features...
  C columns found: ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14']
  test shape: (506691, 447)
  test memory: 2267.5 MB

Realigning columns...
Final: X=(590540, 447), test=(506691, 447)
Feature engineering done ✓


In [7]:
print("=== FEATURE SELECTION ===")
print(f"Starting shape: X={X.shape}, test={test.shape}")

# ── 1. drop high missing >90% ──
print("\nStep 1: Dropping columns with >90% missing...")
missing_rate = X.isnull().mean()
high_missing = missing_rate[missing_rate > 0.9].index.tolist()
print(f"  Dropped: {len(high_missing)} cols")
print(f"  {high_missing}")
X    = X.drop(columns=high_missing)
test = test.drop(columns=high_missing)
print(f"  After: X={X.shape}")

# ── 2. drop near-zero variance ──
print("\nStep 2: Dropping near-zero variance (std < 0.01)...")
num_cols_temp = X.select_dtypes(include=[np.number]).columns
low_var = [c for c in num_cols_temp if X[c].std() < 0.01]
print(f"  Dropped: {len(low_var)} cols → {low_var}")
X    = X.drop(columns=low_var)
test = test.drop(columns=low_var)
print(f"  After: X={X.shape}")

# ── 3. drop C columns since we extracted their signal already ──
print("\nStep 3: Dropping raw C columns (signal captured in C_mean/C_std/C_max/C_min)...")
c_cols = [c for c in X.columns if c.startswith("C") and c[1:].isdigit()]
print(f"  Dropping: {c_cols}")
X    = X.drop(columns=c_cols)
test = test.drop(columns=c_cols)
print(f"  After: X={X.shape}")

# ── 4. drop highly correlated numeric features (r > 0.95) ──
print("\nStep 4: Dropping highly correlated features (r > 0.95)...")
num_cols_temp = X.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = X[num_cols_temp].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr = [col for col in upper.columns if any(upper[col] > 0.95)]
print(f"  Dropped: {len(high_corr)} cols")
print(f"  {high_corr}")
X    = X.drop(columns=high_corr)
test = test.drop(columns=high_corr)
print(f"  After: X={X.shape}")

# ── 5. drop high cardinality categoricals (>200 unique) ──
print("\nStep 5: Dropping high cardinality categoricals (>200 unique)...")
cat_cols_temp = X.select_dtypes(include=["object"]).columns
high_card = [c for c in cat_cols_temp if X[c].nunique() > 200]
print(f"  Dropped: {len(high_card)} cols → {high_card}")
X    = X.drop(columns=high_card)
test = test.drop(columns=high_card)
print(f"  After: X={X.shape}")

print(f"\n=== FINAL SHAPES AFTER SELECTION ===")
print(f"  X:    {X.shape}")
print(f"  test: {test.shape}")
assert list(X.columns) == list(test.columns)
print("Columns aligned ✓")

=== FEATURE SELECTION ===
Starting shape: X=(590540, 447), test=(506691, 447)

Step 1: Dropping columns with >90% missing...
  Dropped: 12 cols
  ['dist2', 'D7', 'id_07', 'id_08', 'id_18', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27']
  After: X=(590540, 435)

Step 2: Dropping near-zero variance (std < 0.01)...
  Dropped: 3 cols → ['V1', 'V305', 'C_min']
  After: X=(590540, 432)

Step 3: Dropping raw C columns (signal captured in C_mean/C_std/C_max/C_min)...
  Dropping: ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14']
  After: X=(590540, 418)

Step 4: Dropping highly correlated features (r > 0.95)...
  Dropped: 130 cols
  ['D2', 'D6', 'D12', 'V11', 'V16', 'V18', 'V21', 'V22', 'V28', 'V30', 'V32', 'V33', 'V34', 'V43', 'V49', 'V50', 'V52', 'V57', 'V58', 'V60', 'V63', 'V70', 'V71', 'V72', 'V74', 'V81', 'V84', 'V89', 'V91', 'V92', 'V93', 'V94', 'V97', 'V101', 'V102', 'V103', 'V106', 'V126', 'V127', 'V128', 'V132', 'V133', 'V134', 'V

In [8]:
print("=== MEMORY REDUCTION ===")
print(f"Before: X={X.memory_usage(deep=True).sum()/1024**2:.1f} MB, test={test.memory_usage(deep=True).sum()/1024**2:.1f} MB")

def reduce_mem(df, name):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    print(f"  {name}: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")
    return df

X    = reduce_mem(X,    "X")
test = reduce_mem(test, "test")

print(f"\nAfter:")
print(f"  X:    {X.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print(f"  test: {test.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print(f"\nColumn dtypes summary:")
print(X.dtypes.value_counts())

=== MEMORY REDUCTION ===
Before: X=1816.8 MB, test=1569.3 MB
  X: 1257.0 MB
  test: 1089.0 MB

After:
  X:    1257.0 MB
  test: 1089.0 MB

Column dtypes summary:
float32    253
object      29
int32        1
int16        1
int8         1
Name: count, dtype: int64


**MLflow / DagsHub setup**

In [10]:
print("Setting up MLflow + DagsHub...")
os.environ["MLFLOW_TRACKING_USERNAME"] = "kende23"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "be7a74e194c6b7c0d666a4938cfee49142c7d603"
mlflow.set_tracking_uri("https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow")
dagshub.init(repo_owner="kende23", repo_name="ieee-cis-fraud-detection", mlflow=True)

experiment_name = "xgboost_experiments"
if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(experiment_name)
    print(f"Created experiment: {experiment_name}")
else:
    print(f"Found existing experiment: {experiment_name}")

mlflow.set_experiment(experiment_name)
print("MLflow + DagsHub connected ✓")

Setting up MLflow + DagsHub...


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=0b5bd9fb-f48c-4a21-9fc1-4781ede4cc61&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=132e4f0f5d70d3bcfc44f17cbbe13cf6cf063e0c34eddf4011c59a5bf973cb33




Accessing as kende23

Initialized MLflow to track repo "kende23/ieee-cis-fraud-detection"

Repository kende23/ieee-cis-fraud-detection initialized!

Found existing experiment: xgboost_experiments
MLflow + DagsHub connected ✓


**Split**

In [11]:
print("=== TRAIN/VAL SPLIT ===")
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"  Train fraud rate: {y_train.mean():.4f}")
print(f"  Val   fraud rate: {y_val.mean():.4f}")

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
print(f"\n  Numeric cols:     {len(num_cols)}")
print(f"  Categorical cols: {len(cat_cols)}")
print(f"  Cat cols are:     {cat_cols}")

del X
gc.collect()
print("\nFreed X from memory ✓")

=== TRAIN/VAL SPLIT ===
  X_train: (472432, 285), y_train: (472432,)
  X_val:   (118108, 285),   y_val:   (118108,)
  Train fraud rate: 0.0350
  Val   fraud rate: 0.0350

  Numeric cols:     256
  Categorical cols: 29
  Cat cols are:     ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'P_email_provider', 'R_email_provider']

Freed X from memory ✓


**Build XGBoost pipeline & train**

In [12]:
print("=== BUILDING XGBOOST PIPELINE ===")

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

fraud_count      = int(y_train.sum())
nonfraud_count   = int((y_train == 0).sum())
scale_pos_weight = nonfraud_count / fraud_count
print(f"  scale_pos_weight: {scale_pos_weight:.2f} (fraud={fraud_count}, non-fraud={nonfraud_count})")

model_xgb = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="auc",
        tree_method="hist",
        device="cuda" if os.path.exists("/usr/local/cuda") else "cpu",
        n_jobs=-1
    ))
])

print("Pipeline built ✓")
print("Training XGBoost... (this may take 5-10 minutes)")
model_xgb.fit(X_train, y_train)
print("Training done ✓")

del X_train
gc.collect()
print("Freed X_train from memory ✓")

=== BUILDING XGBOOST PIPELINE ===
  scale_pos_weight: 27.58 (fraud=16530, non-fraud=455902)
Pipeline built ✓
Training XGBoost... (this may take 5-10 minutes)
Training done ✓
Freed X_train from memory ✓


**Evaluate & log**

In [13]:
print("=== EVALUATING XGBOOST ===")

val_probs = model_xgb.predict_proba(X_val)[:, 1]
val_preds = (val_probs >= 0.5).astype(int)

val_auc       = roc_auc_score(y_val, val_probs)
val_precision = precision_score(y_val, val_preds)
val_recall    = recall_score(y_val, val_preds)
val_f1        = f1_score(y_val, val_preds)

print(f"\n=== VALIDATION METRICS ===")
print(f"  AUC:       {val_auc:.4f}")
print(f"  Precision: {val_precision:.4f}")
print(f"  Recall:    {val_recall:.4f}")
print(f"  F1:        {val_f1:.4f}")

metrics = {
    "val_auc":       val_auc,
    "val_precision": val_precision,
    "val_recall":    val_recall,
    "val_f1":        val_f1,
}

params = {
    "n_estimators":          300,
    "max_depth":             6,
    "learning_rate":         0.05,
    "subsample":             0.8,
    "colsample_bytree":      0.8,
    "scale_pos_weight":      round(scale_pos_weight, 2),
    "missing_threshold":     0.9,
    "corr_threshold":        0.95,
    "feature_engineering":   "log_amt, time_features, card_agg, C_agg, email_provider",
}

print("\nLogging to MLflow / DagsHub...")
with mlflow.start_run(run_name="xgboost_with_feature_engineering"):
    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    joblib.dump(model_xgb, "xgb_model.pkl")
    mlflow.log_artifact("xgb_model.pkl")
    print(f"  Run ID: {mlflow.active_run().info.run_id}")

print("Logged to DagsHub ✓")
print("Check: https://dagshub.com/kende23/ieee-cis-fraud-detection")

del X_val
gc.collect()
print("Freed X_val from memory ✓")

=== EVALUATING XGBOOST ===

=== VALIDATION METRICS ===
  AUC:       0.9329
  Precision: 0.2290
  Recall:    0.8079
  F1:        0.3569

Logging to MLflow / DagsHub...
  Run ID: 761b0aa3fdce4184831fa87cab38e893
🏃 View run xgboost_with_feature_engineering at: https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow/#/experiments/2/runs/761b0aa3fdce4184831fa87cab38e893
🧪 View experiment at: https://dagshub.com/kende23/ieee-cis-fraud-detection.mlflow/#/experiments/2
Logged to DagsHub ✓
Check: https://dagshub.com/kende23/ieee-cis-fraud-detection
Freed X_val from memory ✓


**Submission**

In [14]:
print("=== GENERATING SUBMISSION ===")
print(f"  test shape: {test.shape}")

print("Predicting...")
test_probs = model_xgb.predict_proba(test)[:, 1]

print(f"  Predictions shape: {test_probs.shape}")
print(f"  Predicted fraud rate: {test_probs.mean():.4f}")
print(f"  Min prob: {test_probs.min():.4f}")
print(f"  Max prob: {test_probs.max():.4f}")

submission = pd.DataFrame({
    "TransactionID": test_tr["TransactionID"].values,
    "isFraud": test_probs
})

print(f"\nSubmission shape: {submission.shape}")
print(submission.head(10))

submission.to_csv("submission.csv", index=False)
print("\nsubmission.csv saved ✓")
print("Submit at: https://www.kaggle.com/competitions/ieee-fraud-detection/submissions")

=== GENERATING SUBMISSION ===
  test shape: (506691, 285)
Predicting...
  Predictions shape: (506691,)
  Predicted fraud rate: 0.2492
  Min prob: 0.0010
  Max prob: 0.9995

Submission shape: (506691, 2)
   TransactionID   isFraud
0        3663549  0.100876
1        3663550  0.123395
2        3663551  0.128131
3        3663552  0.040301
4        3663553  0.115232
5        3663554  0.139160
6        3663555  0.355622
7        3663556  0.533891
8        3663557  0.030216
9        3663558  0.243095

submission.csv saved ✓
Submit at: https://www.kaggle.com/competitions/ieee-fraud-detection/submissions
